In [ ]:
#ICD9 Depression Coding - Explainable AI Model
#A project where you train a binary classifier to detect ICD-9 code 311 (depression) from discharge notes,
#then explain why the model made its decision by highlighting or surfacing the relevant parts of the text.

# Must have a GPU


In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == 

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

In [ ]:
# Step 1: Load NOTEEVENTS.csv
# Since NOTEEVENTS.csv is too large to be uploaded into google colab, I uploaded into my personal drive so I can call it every time

from google.colab import drive
drive.mount('/content/drive')
location_for_notes = '/content/drive/My Drive/NOTEEVENTS.csv'

dtype_dict = {
    'COLUMN_NAME': 'category',
    # add other columns as needed
}

# Read NOTEEVENTS.csv into chunks:
notes_chunks = pd.read_csv(
    location_for_notes,
    usecols=['SUBJECT_ID', 'HADM_ID', 'CATEGORY', 'TEXT'],
    chunksize=100000,
    low_memory=False
)

# Only keep discharge summaries
notes_list = []
for chunk in notes_chunks:
    filtered_chunk = chunk[chunk['CATEGORY'] == 'Discharge summary']
    notes_list.append(filtered_chunk)

notes = pd.concat(notes_list, ignore_index=True)
del notes_chunks, notes_list  # free RAM immediately


Mounted at /content/drive


In [ ]:
# Step 2: Load and merge with ADMISSIONS and DIAGNOSES_ICD
admissions = pd.read_csv('ADMISSIONS.csv', usecols=['SUBJECT_ID', 'HADM_ID', 'DIAGNOSIS', 'ADMISSION_TYPE'])
diagnoses_icd = pd.read_csv('DIAGNOSES_ICD.csv', usecols=['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE'])

# Merge efficiently after filtering
df = notes.merge(admissions, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
df = df.merge(diagnoses_icd, on=['SUBJECT_ID', 'HADM_ID'], how='inner')

# Immediately clear unused dataframes to save memory
del notes, admissions, diagnoses_icd

In [ ]:
# Step 3: Clean text and create binary label for 311
def clean_text(text):
    text = re.sub(r'\[\*\*.*?\*\*\]', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

#Split into 311 or non-311
patient_notes = df.groupby(['SUBJECT_ID', 'HADM_ID', 'TEXT'])['ICD9_CODE'].apply(set).reset_index()
patient_notes['has_311'] = patient_notes['ICD9_CODE'].apply(lambda codes: 1 if '311' in codes else 0)
patient_notes['clean_text'] = patient_notes['TEXT'].apply(clean_text)


In [ ]:
# === Balance: keep all 311s, sample same number of non-311s ===
depression_df = patient_notes[patient_notes['has_311'] == 1]
non_311_df = patient_notes[patient_notes['has_311'] == 0]


# === Balance: keep 700 311s, sample 700 non-311s ===
# depression_df = patient_notes[patient_notes['has_311'] == 1].sample(n=700, random_state=42)
# non_311_df = patient_notes[patient_notes['has_311'] == 0].sample(n=700, random_state=42)

non_311_sampled = non_311_df.sample(n=len(depression_df), random_state=42)
balanced_df = pd.concat([depression_df, non_311_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ Balanced dataset created:")
print("311 samples:", len(depression_df))
print("Non-311 samples:", len(non_311_sampled))

✅ Balanced dataset created:
311 samples: 3667
Non-311 samples: 3667


In [ ]:
# Step 6: Split into train/test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced_df["clean_text"].tolist(),
    balanced_df["has_311"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
# Step 7: Tokenization using DischargeSummaryBERT
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_Discharge_Summary_BERT")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(tokenize, batched=True)
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels}).map(tokenize, batched=True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/5867 [00:00<?, ? examples/s]

Map:   0%|          | 0/1467 [00:00<?, ? examples/s]

In [ ]:
# Step 8: Load model


model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_Discharge_Summary_BERT",
    num_labels=2,
    problem_type="single_label_classification"
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_Discharge_Summary_BERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Step 9: Trainer
import os
os.environ["WANDB_DISABLED"] = "true" #to prevent authentication when training with Hugging
from transformers import EarlyStoppingCallback


training_args = TrainingArguments(
    output_dir="./results_311",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none", #also to prevent authentication when training with HuggingFace
    learning_rate=4e-6,
    weight_decay=0.1,

)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Step 10: Train and evaluate
trainer.train()

# Save model
model_path = "/content/drive/MyDrive/clinical_icd9_model_8epochs/"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.606800,0.603616
2,0.490600,0.471729
3,0.452600,0.480916
4,0.455600,0.483625
5,0.408600,0.479341
6,0.402200,0.496245
7,0.321000,0.510516
8,0.360900,0.513619


('/content/drive/MyDrive/clinical_icd9_model_8epochs/tokenizer_config.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs/special_tokens_map.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs/vocab.txt',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs/added_tokens.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs/tokenizer.json')

In [ ]:
import os
os.environ["USE_SAFETENSORS"] = "0"

model_path = "/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

('/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/tokenizer_config.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/special_tokens_map.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/vocab.txt',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/added_tokens.json',
 '/content/drive/MyDrive/clinical_icd9_model_8epochs_v2/tokenizer.json')

In [ ]:
# Path to where model and tokenizer are saved
model_path = "/content/drive/MyDrive/clinical_icd9_model_8epochs"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def predict_311_(text, threshold=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=512).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    pred_label = 1 if probs[1] > threshold else 0
    return pred_label, probs

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

preds = []
probs_list = []

for text in test_texts:
    pred, probs = predict_311_(text)
    preds.append(pred)
    probs_list.append(probs[1])  # store probability of class 1 (311)

# Compute metrics
true_labels = test_dataset["label"]

accuracy = accuracy_score(true_labels, preds)
precision = precision_score(true_labels, preds)
recall = recall_score(true_labels, preds)
f1 = f1_score(true_labels, preds)

print("=== Performance Metrics ===")
print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")
print("\nDetailed Report:\n", classification_report(true_labels, preds, target_names=["Not 311", "311"]))


In [ ]:
predict_311("The patient has depression")